# Praktikum 7 NLP
Nicholas Reymond Sihite - 13522144

In [1]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.3/64.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.7/358.7 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.4/284.4 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 18.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      

## Imports

In [2]:
from unsloth import FastLanguageModel
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig, Trainer, TrainingArguments, BitsAndBytesConfig
import transformers
import torch
import torch.nn as nn
from peft import LoraConfig, get_peft_model, PeftModel
import os
from trl import SFTTrainer
from google.colab import files
import shutil
from collections import Counter
import string
import re
from tqdm import tqdm
os.environ["WANDB_DISABLED"] = "true"

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## Load Dataset

In [3]:
ds = load_dataset("jakartaresearch/indoqa", revision="refs/convert/parquet")
df = ds["train"].to_pandas()
df["answer"] = df["answer"].fillna("Tidak dapat dijawab dari konteks.") # Ada yg unanswerable

print(df.head(10))

default/train/0000.parquet:   0%|          | 0.00/1.37M [00:00<?, ?B/s]

default/validation/0000.parquet:   0%|          | 0.00/466k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

                             id  \
0     629HsNl5Qf3THUq_CbmFrSx28   
1  2648F832Z8D-thJnrxy_ZoZ11111   
2  2561YDPNx6Nm5o6T0F-PGw-u2466   
3   1431C0_ORK0ta8pDX6H-1VOp596   
4  1234WIzl7ElikCCPk4gA-7ua3057   
5    580J0GQq9CMPFwuNoK_eRB4169   
6            63292Pkbg0_0W_4375   
7   1008cPycwBnuEuEiZLC7k7LH387   
8  2515HnCK6br3syHDxNhRpc8g1750   
9   1423Rcv2EkrmFuIt2YeOnjUf618   

                                             context  \
0  Para pemuda pejuang, termasuk Chaerul Saleh, S...   
1  Malaysia terdiri atas dua kawasan utama yang t...   
2  Pada 16 Februari 1830, Diponegoro setuju untuk...   
3  Kerajaan Buddhis Ligor segera kembali mengenda...   
4  Sikap terhadap perbudakan juga berubah; semua ...   
5  Lembaga perwakilan tertinggi yaitu Majelis Per...   
6  Presiden Indonesia (nama jabatan resmi: Presid...   
7  Di Amerika Utara, pemerintahan perwakilan terb...   
8  Israel (;) adalah sebuah negara di Timur Tenga...   
9  Malaysia sebagai negara federal tidak pernah a...  

## Load Model

In [4]:
MAX_SEQ = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

==((====))==  Unsloth 2025.11.4: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

## Set Up LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.11.3 patched 16 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


## Check Trainable Params

In [ ]:
def print_trainable_parameters(model):
    """
  printing the number of trainable paramters in the model
  """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}")

print_trainable_parameters(model)

trainable params: 11272192 || all params: 785713152 || trainable%: 1.434644688243681


## Format Data

In [5]:
alpaca_prompt = """Berdasarkan konteks berikut, jawab pertanyaan dengan singkat dan tepat (maksimal 5 kata jika memungkinkan).

### Konteks:
{}

### Pertanyaan:
{}

### Jawaban:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_func(examples):
    contexts = examples["context"]
    questions = examples["question"]
    answers = examples["answer"]
    texts = []
    for context, question, answer in zip(contexts, questions, answers):
        text = alpaca_prompt.format(context, question, answer) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

dataset = ds["train"].map(formatting_func, batched=True)

Map:   0%|          | 0/3309 [00:00<?, ? examples/s]

## Fine-tune Model

In [ ]:
# training_args = TrainingArguments(
#     output_dir="indoqa-lora-1b",
#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=8,
#     warmup_steps=50,
#     max_steps=2000,
#     learning_rate=2e-4,
#     fp16=not torch.cuda.is_bf16_supported(),
#     bf16=torch.cuda.is_bf16_supported(),
#     logging_steps=20,
#     save_steps=500,
#     optim="adamw_8bit",
#     weight_decay=0.01,
#     lr_scheduler_type="linear",
#     seed=42,
#     report_to="none",
# )

training_args = TrainingArguments(
    output_dir="indoqa-lora-1b",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=50,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=20,
    save_steps=400,
    save_total_limit=2,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ,
    packing=False,
    dataset_kwargs={
        "add_special_tokens": False,
        "append_concat_token": False,
    },
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/3309 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,309 | Num Epochs = 3 | Total steps = 1,242
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
20,2.234200
40,1.904700
60,1.804100
80,1.811600
100,1.756600
120,1.688000
140,1.693000
160,1.699500
180,1.597200
200,1.656700


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


TrainOutput(global_step=1242, training_loss=1.160219537847092, metrics={'train_runtime': 2046.8289, 'train_samples_per_second': 4.85, 'train_steps_per_second': 0.607, 'total_flos': 1.7447794581037056e+16, 'train_loss': 1.160219537847092, 'epoch': 3.0})

## Save LoRA Adapters

In [ ]:
model.save_pretrained("indoqa-lora-1b")
tokenizer.save_pretrained("indoqa-lora-1b")

('indoqa-lora-1b/tokenizer_config.json',
 'indoqa-lora-1b/special_tokens_map.json',
 'indoqa-lora-1b/chat_template.jinja',
 'indoqa-lora-1b/tokenizer.json')

In [ ]:
folder = "indoqa-lora-1b"
shutil.make_archive(folder, 'zip', folder)
files.download(folder + ".zip")

## Load LoRA for Inference

In [6]:
MAX_SEQ = 512
adapter_path = "indoqa-lora-1b"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=adapter_path,
    max_seq_length=MAX_SEQ,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)

==((====))==  Unsloth 2025.11.4: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2025.11.4 patched 16 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048, padding_idx=128004)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (

## Inference

In [8]:
def normalize_answer(s):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return ""

    s = str(s)

    def remove_articles(text):
        return re.sub(r'\b(a|an|the|di|ke|dari|yang|untuk)\b', ' ', text)

    def white_space_fix(text):
        return ' '.join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punc(lower(s))))

def f1_score(prediction, ground_truth):
    prediction_tokens = normalize_answer(prediction).split()
    ground_truth_tokens = normalize_answer(ground_truth).split()

    if len(prediction_tokens) == 0 or len(ground_truth_tokens) == 0:
        return int(prediction_tokens == ground_truth_tokens)

    common = Counter(prediction_tokens) & Counter(ground_truth_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0

    precision = 1.0 * num_same / len(prediction_tokens)
    recall = 1.0 * num_same / len(ground_truth_tokens)
    f1 = (2 * precision * recall) / (precision + recall)

    return f1

def exact_match_score(prediction, ground_truth):
    return normalize_answer(prediction) == normalize_answer(ground_truth)

val_dataset = ds["validation"]

def evaluate_model(model, tokenizer, dataset, num_samples=None):
    if num_samples is not None:
        dataset = dataset.select(range(min(num_samples, len(dataset))))

    exact_matches = []
    f1_scores = []

    print(f"\nEvaluating on {len(dataset)} samples...")

    for i, example in enumerate(tqdm(dataset)):
        context = example["context"]
        question = example["question"]
        ground_truth = example["answer"]

        prompt = alpaca_prompt.format(context, question, "")
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=30,
                temperature=0.1,
                top_p=0.9,
                do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        response = tokenizer.decode(outputs[0], skip_special_tokens=True)

        if "### Jawaban:" in response:
            prediction = response.split("### Jawaban:")[-1].strip()
        else:
            prediction = response.strip()

        em = exact_match_score(prediction, ground_truth)
        f1 = f1_score(prediction, ground_truth)

        exact_matches.append(em)
        f1_scores.append(f1)

    avg_em = sum(exact_matches) / len(exact_matches) * 100
    avg_f1 = sum(f1_scores) / len(f1_scores) * 100

    print(f"\n{'='*70}")
    print(f"EVALUATION RESULTS:")
    print(f"{'='*70}")
    print(f"Exact Match (EM): {avg_em:.2f}%")
    print(f"F1 Score: {avg_f1:.2f}%")
    print(f"Total samples: {len(dataset)}")
    print(f"{'='*70}")

    return {
        "exact_match": avg_em,
        "f1": avg_f1,
        "num_samples": len(dataset)
    }

results = evaluate_model(
    model,
    tokenizer,
    val_dataset,
    num_samples=None
)


Evaluating on 1104 samples...


100%|██████████| 1104/1104 [09:13<00:00,  1.99it/s]


EVALUATION RESULTS:
Exact Match (EM): 51.36%
F1 Score: 73.68%
Total samples: 1104
